# YESNO and VOiCES: Audio Classification, Event Detection, and Voice Activity Detection

**Student:** Abdulameer Farhan Aljmmali  
**Student ID:** 404131110  
**Course:** Speech and Audio Processing

This notebook is written for Google Colab. Run the cells from top to bottom.

**Tasks**
1. Train and compare several classifiers on the YESNO dataset (`torchaudio`).
2. Explore the VOiCES dataset and classify recordings by acoustic distractor/condition when the metadata provides labels.
3. Demonstrate Voice Activity Detection (VAD) with WebRTC VAD and check it against reference speech segments.

> Note: a recording-level label is not a time boundary. If VOiCES only gives recording-level distractor labels, task 2 is recording-level acoustic-condition classification, not temporal event localisation.

## 1. Environment setup

In [ ]:
# Install required packages in Google Colab
!pip -q install torchaudio librosa soundfile scikit-learn pandas matplotlib seaborn webrtcvad-wheels

In [ ]:
import os, glob, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa
import torch
import torchaudio
from IPython.display import display

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)

## 2. YESNO dataset: loading and inspection

In [ ]:
# YESNO: 60 recordings at 8 kHz; each contains 8 spoken words ("yes"=1 / "no"=0).
os.makedirs("./data", exist_ok=True)
yesno = torchaudio.datasets.YESNO(root="./data", download=True)
print("Number of recordings:", len(yesno))
waveform, sample_rate, labels = yesno[0]
print("Waveform shape:", tuple(waveform.shape))
print("Sample rate:", sample_rate)
print("Labels:", labels)

plt.figure(figsize=(12, 3))
plt.plot(np.arange(waveform.shape[-1]) / sample_rate, waveform.squeeze().numpy())
plt.xlabel("Time (seconds)"); plt.ylabel("Amplitude")
plt.title("Example YESNO waveform"); plt.grid(True); plt.show()

## 3. Baseline: one feature vector per recording

The simplest approach summarises a whole recording (mean/std of MFCCs etc.) and predicts all 8 labels at once. Averaging over the whole clip throws away *word order*, so this baseline is expected to perform close to chance. It is kept only for comparison with the word-level method in section 4.

In [ ]:
def extract_audio_features(y, sr, n_mfcc=13):
    """Fixed-length summary features for a mono numpy waveform."""
    y = np.asarray(y, dtype=np.float32)
    if len(y) < 512:
        y = np.pad(y, (0, 512 - len(y)))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=128)
    delta = librosa.feature.delta(mfcc, width=3) if mfcc.shape[1] >= 3 else np.zeros_like(mfcc)
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=128)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=512, hop_length=128)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=512, hop_length=128)[0]
    return np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1), delta.mean(axis=1),
        [rms.mean(), rms.std(), zcr.mean(), zcr.std(), centroid.mean(), len(y) / sr]
    ]).astype(np.float32)

def make_models():
    return {
        "KNN": make_pipeline(SimpleImputer(), StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
        "SVM": make_pipeline(SimpleImputer(), StandardScaler(), SVC(kernel="rbf", C=1.0)),
        "Decision Tree": make_pipeline(SimpleImputer(), DecisionTreeClassifier(random_state=SEED)),
        "Random Forest": make_pipeline(SimpleImputer(), RandomForestClassifier(n_estimators=300, random_state=SEED)),
        "Logistic Regression": make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000)),
    }

recordings = [(w.squeeze().numpy(), sr, np.asarray(t, dtype=int)) for w, sr, t in yesno]
X_rec = np.vstack([extract_audio_features(y, sr) for y, sr, _ in recordings])
Y_rec = np.vstack([t for _, _, t in recordings])
print("Recording-level features:", X_rec.shape, "targets:", Y_rec.shape)

# 5-fold cross-validation (more reliable than one split on only 60 recordings)
cv_rec = KFold(n_splits=5, shuffle=True, random_state=SEED)
baseline_rows = []
for name, est in make_models().items():
    pred = cross_val_predict(MultiOutputClassifier(est), X_rec, Y_rec, cv=cv_rec)
    baseline_rows.append({"Classifier": name,
                          "Word Accuracy": np.mean(pred == Y_rec),
                          "Exact Match (all 8)": np.mean(np.all(pred == Y_rec, axis=1))})
baseline_df = pd.DataFrame(baseline_rows).sort_values("Word Accuracy", ascending=False)
display(baseline_df)

## 4. Improved method: segment each recording into words

Each YESNO clip contains exactly 8 words separated by silence. We find the 8 word regions with short-time energy, extract features **per word**, and train a single binary classifier (*yes* vs *no*). The recording's 8-label prediction is then the sequence of its 8 word predictions.

To avoid leakage, words from the same recording are never split between training and test folds (`GroupKFold` by recording).

In [ ]:
import numpy as np

def segment_words(y, sr, n_words=8, frame_ms=20, hop_ms=10,
                  min_gap_s=0.10, min_word_s=0.08):
    """Split a YESNO recording into exactly `n_words` (start, end) sample ranges.

    Uses short-time energy with an adaptive threshold, merges short gaps,
    drops very short blips, then forces exactly n_words regions by merging the
    closest neighbours (too many) or splitting the longest region (too few).
    """
    y = np.asarray(y, dtype=np.float32)
    frame = int(sr * frame_ms / 1000)
    hop = int(sr * hop_ms / 1000)
    if len(y) < frame:
        return []
    n_frames = 1 + (len(y) - frame) // hop
    idx = np.arange(frame)[None, :] + hop * np.arange(n_frames)[:, None]
    energy = np.sqrt(np.mean(y[idx] ** 2, axis=1))
    log_e = 20 * np.log10(energy + 1e-10)

    # Adaptive threshold between the noise floor and the speech level
    floor, peak = np.percentile(log_e, 10), np.percentile(log_e, 95)
    active = log_e > floor + 0.35 * (peak - floor)

    # Contiguous active runs -> (start_frame, end_frame) inclusive-exclusive
    padded = np.concatenate([[False], active, [False]]).astype(int)
    edges = np.flatnonzero(np.diff(padded))
    regions = [[s, e] for s, e in zip(edges[::2], edges[1::2])]

    # Merge regions separated by gaps shorter than min_gap_s
    min_gap = int(min_gap_s * 1000 / hop_ms)
    merged = []
    for r in regions:
        if merged and r[0] - merged[-1][1] < min_gap:
            merged[-1][1] = r[1]
        else:
            merged.append(r)
    # Drop blips shorter than min_word_s
    min_len = int(min_word_s * 1000 / hop_ms)
    regions = [r for r in merged if r[1] - r[0] >= min_len]
    if not regions:
        return []

    # Force exactly n_words regions
    while len(regions) > n_words:
        gaps = [regions[i + 1][0] - regions[i][1] for i in range(len(regions) - 1)]
        i = int(np.argmin(gaps))
        regions[i] = [regions[i][0], regions[i + 1][1]]
        del regions[i + 1]
    while len(regions) < n_words:
        i = int(np.argmax([r[1] - r[0] for r in regions]))
        s, e = regions[i]
        mid = (s + e) // 2
        regions[i:i + 1] = [[s, mid], [mid, e]]

    return [(s * hop, min(len(y), e * hop + frame)) for s, e in regions]

In [ ]:
# Visual check of the segmentation on one recording
y0, sr0, t0 = recordings[0]
segs0 = segment_words(y0, sr0)
plt.figure(figsize=(12, 3))
plt.plot(np.arange(len(y0)) / sr0, y0, lw=0.6)
for (s, e), lab in zip(segs0, t0):
    plt.axvspan(s / sr0, e / sr0, color="tab:green" if lab else "tab:red", alpha=0.25)
    plt.text((s + e) / 2 / sr0, y0.max() * 0.9, "yes" if lab else "no", ha="center")
plt.xlabel("Time (seconds)"); plt.title("Detected word segments (green = yes, red = no)")
plt.grid(True); plt.show()

In [ ]:
X_words, y_words, groups, word_pos = [], [], [], []
for rec_id, (y, sr, target) in enumerate(recordings):
    segs = segment_words(y, sr, n_words=len(target))
    assert len(segs) == len(target), f"recording {rec_id}: found {len(segs)} segments"
    for pos, ((s, e), lab) in enumerate(zip(segs, target)):
        X_words.append(extract_audio_features(y[s:e], sr))
        y_words.append(lab); groups.append(rec_id); word_pos.append(pos)

X_words = np.vstack(X_words)
y_words = np.asarray(y_words); groups = np.asarray(groups)
print("Word-level features:", X_words.shape)
print("Yes / No counts:", np.bincount(y_words)[::-1])

In [ ]:
cv_words = GroupKFold(n_splits=5)
word_rows, word_preds = [], {}
for name, est in make_models().items():
    pred = cross_val_predict(est, X_words, y_words, cv=cv_words, groups=groups)
    word_preds[name] = pred
    # Rebuild each recording's 8-label sequence to compute exact match
    pred_seq = pred.reshape(len(recordings), -1)
    true_seq = y_words.reshape(len(recordings), -1)
    word_rows.append({"Classifier": name,
                      "Word Accuracy": accuracy_score(y_words, pred),
                      "Word F1": f1_score(y_words, pred),
                      "Exact Match (all 8)": np.mean(np.all(pred_seq == true_seq, axis=1))})
word_df = pd.DataFrame(word_rows).sort_values(["Word Accuracy", "Exact Match (all 8)"], ascending=False)
display(word_df)

In [ ]:
# Compare baseline vs word-level method
comp = pd.concat([baseline_df.assign(Method="Whole recording (baseline)"),
                  word_df.assign(Method="Word segmentation")])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=comp, x="Classifier", y="Word Accuracy", hue="Method", ax=axes[0])
axes[0].set_ylim(0, 1.05); axes[0].set_title("YESNO: word-level accuracy (5-fold CV)")
axes[0].tick_params(axis="x", rotation=30)
sns.barplot(data=comp, x="Classifier", y="Exact Match (all 8)", hue="Method", ax=axes[1])
axes[1].set_ylim(0, 1.05); axes[1].set_title("YESNO: exact match of all 8 labels")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

best_name = word_df.iloc[0]["Classifier"]
print("Best word-level classifier:", best_name)
print(classification_report(y_words, word_preds[best_name], target_names=["no", "yes"]))
cm = confusion_matrix(y_words, word_preds[best_name])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["no", "yes"], yticklabels=["no", "yes"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"YESNO confusion matrix ({best_name})"); plt.show()

### YESNO evaluation notes
- **Word Accuracy**: share of individual yes/no words classified correctly.
- **Exact Match**: a recording counts as correct only if all 8 words are correct.
- All scores come from 5-fold cross-validation, so every recording is tested exactly once.
- The baseline shows why temporal structure matters: averaging features over the whole clip removes word order, so the 8 positional labels cannot be recovered. Segmenting first turns the problem into a simple binary task.

## 5. VOiCES dataset: set dataset paths

In [ ]:
# If the VOiCES devkit is already extracted in Colab, use the path below.
# Based on the commonly used extraction layout:
VOICES_ROOT = "/content/VOiCES/VOiCES_devkit"
TRAIN_INDEX = os.path.join(VOICES_ROOT, "references", "train_index.csv")
TEST_INDEX  = os.path.join(VOICES_ROOT, "references", "test_index.csv")

print("VOiCES root exists:", os.path.isdir(VOICES_ROOT))
print("Train index exists:", os.path.isfile(TRAIN_INDEX))
print("Test index exists:", os.path.isfile(TEST_INDEX))

# If your files are stored elsewhere, change VOICES_ROOT above.

In [ ]:
if not os.path.isfile(TRAIN_INDEX):
    print("VOiCES index CSV was not found.")
    print("Upload/extract the VOiCES devkit, then update VOICES_ROOT.")
else:
    train_meta = pd.read_csv(TRAIN_INDEX)
    test_meta = pd.read_csv(TEST_INDEX) if os.path.isfile(TEST_INDEX) else pd.DataFrame()
    print("Train metadata shape:", train_meta.shape)
    print("Train columns:", train_meta.columns.tolist())
    display(train_meta.head())
    if not test_meta.empty:
        print("Test metadata shape:", test_meta.shape)
        display(test_meta.head())

## 6. VOiCES metadata inspection and label/path selection

In [ ]:
def find_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        for low, original in lower_map.items():
            if candidate in low:
                return original
    return None

if 'train_meta' in globals() and not train_meta.empty:
    label_col = find_column(train_meta.columns, ["distractor", "noise", "condition", "event", "class", "label", "type"])
    path_col = find_column(train_meta.columns, ["audio_path", "wav", "file", "path", "filename"])
    print("Candidate label column:", label_col)
    print("Candidate audio/path column:", path_col)
    if label_col:
        print("Label distribution:")
        display(train_meta[label_col].value_counts(dropna=False).to_frame("count"))
    else:
        print("No obvious class-label column found. Inspect the columns above and set label_col manually.")
    if path_col:
        print("Example path values:")
        display(train_meta[path_col].head(10))
else:
    print("Load the VOiCES train index first.")

## 7. VOiCES recording-level event/distractor classification

This section uses the class label available in the index file and audio-level features (MFCC, spectral centroid, bandwidth, ZCR, RMS). The code deliberately checks the metadata columns instead of assuming a fixed schema.

If the automatic column detection does not select the correct columns, set `label_col` and `path_col` manually after inspecting the printed column names. The label should represent an acoustic distractor/condition category.

In [ ]:
# Set these manually if automatic detection selected the wrong columns.
# Example:
# label_col = "distractor"
# path_col = "audio_path"

def resolve_audio_path(value, root):
    value = str(value)
    if os.path.isabs(value) and os.path.isfile(value):
        return value
    candidates = [
        os.path.join(root, value),
        os.path.join(os.path.dirname(root), value),
        os.path.join(root, "audio", value),
    ]
    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate
    # Search by basename as a fallback (can be slow on a large corpus)
    basename = os.path.basename(value)
    matches = glob.glob(os.path.join(root, "**", basename), recursive=True)
    return matches[0] if matches else None

def extract_file_features(path, target_sr=16000):
    y, sr = sf.read(path)
    if y.ndim == 2:
        y = np.mean(y, axis=1)
    y = y.astype(np.float32)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]
    return np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        [np.mean(centroid), np.std(centroid),
         np.mean(bandwidth), np.mean(rolloff),
         np.mean(zcr), np.std(zcr), np.mean(rms), np.std(rms)]
    ]).astype(np.float32)

voices_features = None
voices_labels = None
voices_table = None

if 'train_meta' in globals() and not train_meta.empty and label_col and path_col:
    work = train_meta[[label_col, path_col]].dropna().copy()
    # Limit the number of files for a manageable first experiment.
    # Increase MAX_FILES after confirming the pipeline works.
    MAX_FILES = 400
    work = work.groupby(label_col, group_keys=False).head(
        max(1, MAX_FILES // max(1, work[label_col].nunique()))
    )
    work["resolved_path"] = work[path_col].apply(lambda p: resolve_audio_path(p, VOICES_ROOT))
    work = work.dropna(subset=["resolved_path"])
    print("Audio files found:", len(work))
    print("Classes:", work[label_col].nunique())
    display(work[label_col].value_counts().to_frame("count"))

    feature_rows, valid_rows = [], []
    for _, row in work.iterrows():
        try:
            feature_rows.append(extract_file_features(row["resolved_path"]))
            valid_rows.append(row)
        except Exception as exc:
            print("Skipped:", row["resolved_path"], "|", str(exc)[:120])

    if len(feature_rows) > 0:
        voices_features = np.vstack(feature_rows)
        voices_table = pd.DataFrame(valid_rows).reset_index(drop=True)
        voices_labels = voices_table[label_col].astype(str).to_numpy()
        print("VOiCES feature matrix:", voices_features.shape)
        print("VOiCES label vector:", voices_labels.shape)
else:
    print("Set valid label_col/path_col and load train_meta before running this section.")

In [ ]:
if voices_features is not None:
    class_counts = pd.Series(voices_labels).value_counts()
    usable_classes = class_counts[class_counts >= 2].index
    keep = np.isin(voices_labels, usable_classes)
    Xv, yv = voices_features[keep], voices_labels[keep]

    Xv_train, Xv_test, yv_train, yv_test = train_test_split(
        Xv, yv, test_size=0.25, random_state=SEED, stratify=yv
    )
    voices_models = {
        "Random Forest": make_pipeline(SimpleImputer(), RandomForestClassifier(
            n_estimators=250, random_state=SEED, class_weight="balanced"
        )),
        "SVM": make_pipeline(SimpleImputer(), StandardScaler(),
            SVC(kernel="rbf", class_weight="balanced"))
    }

    voices_results = []
    voices_predictions = {}
    for name, model in voices_models.items():
        model.fit(Xv_train, yv_train)
        pred = model.predict(Xv_test)
        voices_predictions[name] = pred
        voices_results.append({
            "Classifier": name,
            "Accuracy": accuracy_score(yv_test, pred),
            "Macro F1": f1_score(yv_test, pred, average="macro", zero_division=0)
        })

    voices_results_df = pd.DataFrame(voices_results).sort_values("Macro F1", ascending=False)
    display(voices_results_df)
else:
    print("VOiCES features are not available. Complete the previous cell first.")

In [ ]:
if voices_features is not None:
    best_voices = voices_results_df.iloc[0]["Classifier"]
    pred = voices_predictions[best_voices]
    print("Best VOiCES model:", best_voices)
    print(classification_report(yv_test, pred, zero_division=0))

    labels_sorted = sorted(np.unique(yv_test))
    cm = confusion_matrix(yv_test, pred, labels=labels_sorted)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_sorted, yticklabels=labels_sorted)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("VOiCES recording-level class confusion matrix")
    plt.tight_layout()
    plt.show()

### VOiCES interpretation
The reported scores apply only to the extracted subset and the selected metadata label. They should not be described as full-dataset results. If the index labels identify distractor/background conditions, the experiment detects the dominant recording-level acoustic condition. It does not identify the exact start/end time of an event.

## 8. Voice Activity Detection (VAD) using WebRTC

In [ ]:
import webrtcvad

def webrtc_vad(signal, sr, aggressiveness=2, frame_ms=30, target_sr=16000):
    """Return per-frame speech flags (bool array) and the frame length in seconds."""
    signal = np.asarray(signal, dtype=np.float32)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    if sr != target_sr:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=target_sr)
    pcm16 = (np.clip(signal, -1.0, 1.0) * 32767).astype(np.int16)
    flen = int(target_sr * frame_ms / 1000)
    vad = webrtcvad.Vad(aggressiveness)
    flags = [vad.is_speech(pcm16[i:i + flen].tobytes(), target_sr)
             for i in range(0, len(pcm16) - flen + 1, flen)]
    return np.asarray(flags, dtype=bool), frame_ms / 1000

# Use a VOiCES recording if section 7 loaded one; otherwise a YESNO recording.
if voices_table is not None and len(voices_table) > 0:
    vad_path = voices_table.iloc[0]["resolved_path"]
    vad_signal, vad_sr = sf.read(vad_path)
    print("VAD input: VOiCES file", vad_path)
else:
    vad_signal, vad_sr, _ = recordings[0]
    print("VAD input: YESNO recording 0 (VOiCES audio not loaded)")

speech_flags, frame_sec = webrtc_vad(vad_signal, vad_sr)
print("Total frames:", len(speech_flags))
print("Speech frames:", int(speech_flags.sum()))
print("Non-speech frames:", int((~speech_flags).sum()))
print("Speech-frame percentage:", round(100 * speech_flags.mean(), 2))

frame_times = np.arange(len(speech_flags)) * frame_sec
t_sig = np.arange(len(vad_signal)) / vad_sr
sig = vad_signal.mean(axis=1) if np.ndim(vad_signal) == 2 else vad_signal
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(t_sig, sig / (np.abs(sig).max() + 1e-9), lw=0.5, color="gray")
ax.step(frame_times, speech_flags.astype(int), where="post", color="tab:blue", label="VAD speech")
ax.set_xlabel("Time (seconds)"); ax.set_title("WebRTC VAD decisions over the waveform")
ax.legend(loc="upper right"); ax.grid(True); plt.show()

### VAD check against reference word segments (YESNO)

YESNO has no official frame-level speech labels, but the 8 word regions from section 4 give a reasonable reference. We use them to compute frame-level precision, recall and F1 for each WebRTC aggressiveness level across all 60 recordings. These are *approximate* scores, since the reference comes from energy segmentation rather than manual annotation.

In [ ]:
vad_rows = []
for aggr in [0, 1, 2, 3]:
    all_true, all_pred = [], []
    for y, sr, target in recordings:
        flags, fsec = webrtc_vad(y, sr, aggressiveness=aggr)
        ref = np.zeros(len(flags), dtype=bool)
        for s, e in segment_words(y, sr, n_words=len(target)):
            ref[int(s / sr / fsec):int(np.ceil(e / sr / fsec))] = True
        all_true.append(ref); all_pred.append(flags)
    t, p = np.concatenate(all_true), np.concatenate(all_pred)
    vad_rows.append({"Aggressiveness": aggr,
                     "Accuracy": accuracy_score(t, p),
                     "Precision": precision_score(t, p, zero_division=0),
                     "Recall": recall_score(t, p, zero_division=0),
                     "F1": f1_score(t, p, zero_division=0)})
vad_df = pd.DataFrame(vad_rows)
display(vad_df)
vad_df.set_index("Aggressiveness")[["Precision", "Recall", "F1"]].plot(kind="bar", figsize=(8, 3.5), ylim=(0, 1.05),
                                                                     title="WebRTC VAD vs reference word segments (YESNO)")
plt.xticks(rotation=0); plt.grid(axis="y"); plt.show()

## 9. Discussion and limitations

1. **YESNO**: whole-recording features cannot recover the order of the 8 words, so the baseline stays near chance. Segmenting into words first makes each word an easy yes/no decision; this is the main result of task 1.
2. YESNO is very small (60 recordings, 480 words), so results were estimated with 5-fold cross-validation, grouped by recording for the word-level models.
3. **VOiCES** results depend on the metadata schema and on the audio being available locally. The code inspects the index and lets the user confirm the label/path columns. Scores apply only to the subset used, not the full corpus.
4. The VOiCES classifier is recording-level: it identifies the dominant acoustic condition, not the start/end time of events.
5. **WebRTC VAD** is a pretrained detector; it is applied, not trained. Its scores on YESNO use energy-based reference segments, so they are approximate. Higher aggressiveness usually trades recall for precision.

## 10. Conclusion
This notebook provides a reproducible Python workflow for: word-level yes/no classification on YESNO (with a whole-recording baseline for comparison), recording-level acoustic-condition classification on VOiCES when suitable labels are present, and speech/non-speech detection with WebRTC VAD evaluated against reference segments. All numbers are produced by running the notebook; none are pre-filled.

**Abdulameer Farhan Aljmmali — 404131110**